# W5 crop-restricted radiomics and LightGBM

**Scientific purpose.** Document the historical four-phase PyRadiomics extraction,
fold-specific feature selection, and LightGBM classification branch.

**Runnability classification:** historical workflow requiring private processed arrays
and patient-level splits. It is not launched automatically and cannot recreate an exact
serialized historical model object that was not retained.

**Inputs:** private crop arrays and patient folds. **Private outputs when run:** crop-restricted
feature tables and fold-level probabilities outside the repository.

Radiomics is computed from the tumor portion retained inside each fixed independently
lesion-centered crop. These are crop-restricted tumor features, not guaranteed
whole-lesion features.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import numpy as np
import pandas as pd
import SimpleITK as sitk
import lightgbm as lgb
from radiomics import featureextractor
from sklearn.feature_selection import f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from liver_tumor_pipeline.config import load_config
from liver_tumor_pipeline.radiomics import validate_radiomics_config

SPLIT_ROOT = PRIVATE_ARTIFACT_ROOT / "splits"
PROCESSED_ROOT = PRIVATE_ARTIFACT_ROOT / "processed" / "internal"
RADIOMICS_CONFIG = load_config(REPO_ROOT / "configs" / "radiomics.yaml")
validate_radiomics_config(RADIOMICS_CONFIG)
if not PROCESSED_ROOT.is_dir() or not (SPLIT_ROOT / "train_val_test.json").is_file():
    raise FileNotFoundError(
        "Private processed arrays or split definitions required by W5 are unavailable."
    )


## Locked radiomics specification and geometry limitation

Cached 0–255 arrays; no additional z-score normalization; bin width 5; three-dimensional
original-image features; shape, first-order, GLCM, GLRLM, GLSZM, GLDM, and NGTDM.
The expected matrix contains 107 features per phase and 428 candidates per patient.

Cached arrays did not retain source geometry or spacing. NumPy arrays are converted to SimpleITK
images with unit spacing, matching the retained workflow. Shape outputs are crop-grid or
voxel-coordinate descriptors, not verified physical-volume or physical-distance measurements.


In [ ]:
PHASE_ORDER = ("P", "C1", "C2", "C3")
FEATURE_CLASSES = ("shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm")


def make_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(
        binWidth=5,
        label=1,
        force2D=False,
        normalize=False,
        resampledPixelSpacing=None,
    )
    extractor.disableAllImageTypes()
    extractor.enableImageTypeByName("Original")
    extractor.disableAllFeatures()
    for feature_class in FEATURE_CLASSES:
        extractor.enableFeatureClassByName(feature_class)
    return extractor


def extract_patient_features(npz_path: Path, extractor) -> dict[str, float | int | str]:
    with np.load(npz_path, allow_pickle=False) as archive:
        images = archive["ct"].astype(np.float32)
        masks = archive["tumor_mask"].astype(np.uint8)
        patient_key = str(archive["patient_id"])
        label = int(archive["label"])
    if images.shape != (4, 96, 96, 96) or masks.shape != images.shape:
        raise ValueError("W5 input arrays do not satisfy the locked four-phase crop shape.")
    row: dict[str, float | int | str] = {"patient_id": patient_key, "label": label}
    for phase_index, phase in enumerate(PHASE_ORDER):
        if masks[phase_index].sum() == 0:
            continue
        image = sitk.GetImageFromArray(images[phase_index])
        mask = sitk.GetImageFromArray(masks[phase_index])
        image.SetSpacing((1.0, 1.0, 1.0))
        mask.SetSpacing((1.0, 1.0, 1.0))
        values = extractor.execute(image, mask)
        for name, value in values.items():
            if name.startswith("diagnostics"):
                continue
            row[f"{phase}_{name}"] = float(value)
    return row


## Fold-specific feature selection and classifier

Zero-variance removal, absolute-correlation pruning at 0.95, univariate ANOVA ranking,
median imputation, and standardization are fit within each fold-training subset.
Thirty features are retained when available. The function returns the fitted components
in memory; it does not claim that serialized historical W5 models were preserved.


In [ ]:
def select_training_features(X, y, names, *, top_k=30, correlation_threshold=0.95):
    standard_deviations = np.nanstd(X, axis=0)
    nonconstant = standard_deviations > 1e-10
    kept_global = np.where(nonconstant)[0]
    X_kept = X[:, nonconstant]
    if X_kept.shape[1] == 0:
        raise ValueError("No nonconstant radiomics features remain.")
    filled = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    correlation = np.abs(np.atleast_2d(np.corrcoef(filled.T)))
    remove = np.zeros(correlation.shape[0], dtype=bool)
    for left in range(correlation.shape[0]):
        if remove[left]:
            continue
        for right in range(left + 1, correlation.shape[0]):
            if not remove[right] and correlation[left, right] > correlation_threshold:
                remove[right] = True
    after_correlation = kept_global[~remove]
    ranked_input = SimpleImputer(strategy="median").fit_transform(X[:, after_correlation])
    scores, _ = f_classif(ranked_input, y)
    order = np.argsort(np.nan_to_num(scores, nan=-np.inf))[::-1]
    selected = after_correlation[order[: min(top_k, len(order))]]
    return selected, [names[index] for index in selected]


def fit_w5_fold(train_frame: pd.DataFrame, validation_frame: pd.DataFrame, feature_names):
    X_train_all = train_frame[feature_names].to_numpy(dtype=np.float32)
    y_train = train_frame["label"].to_numpy()
    X_validation_all = validation_frame[feature_names].to_numpy(dtype=np.float32)
    y_validation = validation_frame["label"].to_numpy()
    selected_indices, selected_names = select_training_features(
        X_train_all, y_train, feature_names
    )
    imputer = SimpleImputer(strategy="median").fit(X_train_all[:, selected_indices])
    X_train = imputer.transform(X_train_all[:, selected_indices])
    X_validation = imputer.transform(X_validation_all[:, selected_indices])
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train)
    X_validation = scaler.transform(X_validation)
    classifier = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        min_child_samples=5,
        reg_alpha=0.1,
        reg_lambda=0.1,
        class_weight="balanced",
        random_state=42,
        verbose=-1,
    )
    classifier.fit(X_train, y_train)
    probabilities = classifier.predict_proba(X_validation)
    predictions = probabilities.argmax(axis=1)
    metrics = {
        "auc": roc_auc_score(y_validation, probabilities, multi_class="ovr", average="macro"),
        "accuracy": accuracy_score(y_validation, predictions),
        "macro_f1": f1_score(y_validation, predictions, average="macro"),
    }
    return {
        "selected_features": selected_names,
        "imputer": imputer,
        "scaler": scaler,
        "classifier": classifier,
        "validation_probabilities": probabilities,
        "metrics": metrics,
    }


## Execution and evidence boundary

Feature extraction and model fitting require authorized patient-level artifacts and can
be expensive. Invoke the functions deliberately in that environment. The reported W5
metrics are preserved as public aggregate values in
`results/aggregate/internal_performance.csv`; patient-level features and predictions are
not released. C2 crop-retention limitations apply to W5, while equivalent aggregate
retention audits for the other phases are unavailable.
